# Module 4 — Auto Loader + Structured Streaming
Exam domain: **Data Processing**

Runs standalone in Google Colab. **Note:** Auto Loader's `cloudFiles` source is a
Databricks-only, cloud-storage-native feature — it does not run outside Databricks.
This notebook simulates the same incremental-ingestion pattern with Structured
Streaming's open-source `file` source, which follows the same core mental model
(new files land in a directory -> stream picks them up -> checkpoint tracks
progress). The real `cloudFiles` syntax is in the Databricks version of this
notebook.

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
import os, shutil, time

builder = (SparkSession.builder
    .appName("Module4-Streaming")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
source_dir = "/content/stream_source"
checkpoint_dir = "/content/stream_checkpoint"
sink_dir = "/content/lake/bronze/events_stream"
shutil.rmtree(source_dir, ignore_errors=True)
shutil.rmtree(checkpoint_dir, ignore_errors=True)
shutil.rmtree(sink_dir, ignore_errors=True)
os.makedirs(source_dir, exist_ok=True)

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("amount", DoubleType()),
])

# Drop the first batch of files before starting the stream
spark.createDataFrame([(1, "Alice", 100.0), (2, "Bob", 50.0)], schema) \
    .write.mode("overwrite").json(f"{source_dir}/batch1")

## Start the stream
`readStream` + `trigger(availableNow=True)` processes everything currently
available and then stops — good for batch-like incremental jobs, which is also
how Auto Loader is commonly scheduled on Databricks.

In [ ]:
stream_df = (spark.readStream
    .schema(schema)
    .json(source_dir))

query = (stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True)
    .start(sink_dir))

query.awaitTermination()
spark.read.format("delta").load(sink_dir).show()

## Simulate a second incremental batch arriving
Drop more files and re-run the same stream: only the new file should be picked
up, because the checkpoint remembers what was already processed.

In [ ]:
spark.createDataFrame([(3, "Cara", 75.0)], schema) \
    .write.mode("overwrite").json(f"{source_dir}/batch2")

query = (stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True)
    .start(sink_dir))
query.awaitTermination()

spark.read.format("delta").load(sink_dir).orderBy("id").show()  # 3 rows total, not 5

## Schema evolution / rescued data
Real Auto Loader can infer schema automatically and evolve it as new columns
appear (`cloudFiles.schemaEvolutionMode`), routing unexpected columns to a
`_rescued_data` column instead of failing the pipeline. With the plain `file`
source used here, schema is fixed up front — see the Databricks version for the
real `cloudFiles` schema inference/evolution options.